# Chap 1 - Construction du jeu de données pour la validation du modèle

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [6]:
titanicDF = pd.read_csv("sample/train.csv")   

In [7]:
# On ignore les ages inconnus

titanicDF.dropna(subset=['Age'], inplace=True)

In [8]:
# encodage du sexe

from sklearn import preprocessing

encoder = preprocessing.LabelEncoder()
titanicDF['encoded_sex'] = encoder.fit_transform(titanicDF['Sex'])

### Validation avec train_test_split

In [9]:
features = ['Age', 'encoded_sex']
X_DF = titanicDF[features]
y_DF = titanicDF['Survived']

# On prend  2/3 des données pour entraîner le modèle (train), et 1/3 pour tester les prédictions (test)

X_trainDF, X_testDF, y_trainDF, y_testDF = train_test_split(X_DF, y_DF, test_size=.33)

In [10]:
print(len(titanicDF))
print(len(X_trainDF))
print(len(y_trainDF))
print(len(X_testDF))
print(len(y_testDF))

714
478
478
236
236


ci-dessous : **exactitude** comme métrique d'évaluation : nb prédictions correctes / nb prédictions

In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics

# n_jobs --> -1 means using all processors. 
forest = RandomForestClassifier(n_estimators=200, n_jobs=-1) 

# Apprentissage sur 2/3 (train)
forest.fit(X=X_trainDF, y=y_trainDF)

# Evaluation : prédiction  sur l'autre 1/3 (test)
y_pred = forest.predict(X_testDF) 

accuracy = metrics.accuracy_score(y_testDF, y_pred)

print("\nExactitude  ", "%.3f" % accuracy)  # The best performance is 1


Exactitude   0.737


### Cross-validation

in a k-fold CV you will be training your model k-times and also testing it k-times. 

In [16]:
from sklearn.model_selection import cross_val_score

newForest = RandomForestClassifier(n_estimators=200, n_jobs=-1) 

# Evaluate a score by cross-validation ; cv--> cross-validation 5 
scores = cross_val_score(newForest, X_DF, y_DF, cv=5)

# cross_val_score fait : 
# 1.  la séparation des données pour entraîner le modèle (train) + pour tester les prédictions (test) - réalisé 5X
# 2.  effectue l'apprentissage 5X sur les différents X-Train X-test créés en 1
# 3.  évalue l'apprentissage à chaque fois (5X)

scores

array([0.77622378, 0.75524476, 0.75524476, 0.73426573, 0.78873239])

In [17]:
np.mean(scores)

0.7619422830690435

Conclusion : fiabilité du modèle qui n'est pas sensible à de nouvelle données. Les scores sont sensiblement les mêmes. 